## Loading the Dataset

In [5]:
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle
import tensorflow as tf 
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout, Multiply, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tqdm import tqdm
import time
from tensorflow.keras.mixed_precision import set_global_policy
import gc
from tensorflow.keras.models import load_model
from tensorflow.keras import layers
import random

In [ ]:
recipe2M = pd.read_csv('recipes_data.csv')

In [ ]:
recipe2M.head()

## Removing unnecessary columns & nulls

In [ ]:
recipe2M_cleaned=recipe2M.drop(columns=['link', 'source', 'site'], inplace=False)
recipe2M_cleaned.dropna()

## Removing recipes with directions contatining the word "step"

In [ ]:

recipe2M_cleaned = recipe2M_cleaned[~recipe2M_cleaned['directions'].str.contains('step', case=False, na=False)]
recipe2M_cleaned['title'].count()


## Removing recipes with at most 1 ingredient

In [ ]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['ingredients'].apply(lambda x: len([i for i in x if i.strip()]) <= 1)].index, inplace=True)
recipe2M_cleaned['title'].count()


## Removing recipes with instructions less than 10 characters

In [ ]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['directions'].apply(lambda x: not all(len(i.strip()) < 10 for i in x if i.strip()))].index, inplace=True)
recipe2M_cleaned['title'].count()

## Removing recipes with title less than 4 characters 

In [ ]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['title'].apply(lambda x: len(str(x)) < 4 if pd.notnull(x) else False)].index, inplace=True)
recipe2M_cleaned['title'].count()

## Extract raw ingredients

In [ ]:
recipes = recipe2M_cleaned

In [ ]:
# Tokenize a string into words
recipes['tokens'] = recipes['NER'].apply(word_tokenize)


In [ ]:
#adding customized stop words
irrelevant_words = {
    'fresh', 'frozen', 'thawed', 'raw', 'grated', 'diced', 'chopped', 'minced',
    'powdered', 'sliced', 'ground', 'cooked', 'boiled', 'roasted', 'steamed',
    'baked', 'fried', 'toasted', 'crushed', 'peeled', 'skinned', 'shredded',
    'melted', 'whipped', 'pinch', 'dash', 'handful', 'cup', 'tablespoon',
    'teaspoon', 'liter', 'ml', 'oz', 'lb', 'gram', 'kg', 'quart', 'optional',
    'to taste', 'as needed', 'prepared', 'ready-made', 'store-bought', 'homemade',
    'pre-cooked', 'large', 'small', 'medium', 'whole', 'half', 'quartered',
    'extra', 'light', 'dark', 'white', 'black', 'red', 'green', 'yellow',
    'brown', 'golden', 'sweet', 'bitter', 'spicy', 'mild', 'hot', 'cold',
    'water', 'broth', 'stock', 'sauce', 'seasoning', 'marinade','bite','size'
}
stop_words = set(stopwords.words('english'))
stop_words.update(irrelevant_words)

In [ ]:
lemmatizer = WordNetLemmatizer()

# Function to lemmatize nouns
def lemmatize(word, pos):
    if pos.startswith('NN'):  
        return lemmatizer.lemmatize(word, pos='n')
    else:
        return word  


In [ ]:
#apply stop words removal and lemmatization
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: [lemmatize(word.lower(), tag) for word, tag in nltk.pos_tag(x) if word.isalnum() and word.lower() not in stop_words]
)

In [ ]:
#filter uninque ingredients
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: list(set(x))
)

In [ ]:
print(recipes['tokens'])

In [ ]:
#adding ids to recipes
recipes['recipe_id'] = recipes.index + 1

In [ ]:
#reorder the columns
columns = ['recipe_id'] + [col for col in recipes.columns if col != 'recipe_id']
recipes = recipes[columns]

In [ ]:
recipes.rename(columns={'tokens': 'raw_ingredients'}, inplace=True)

In [ ]:
recipes.head()

# Extract Cooking Methods

In [ ]:
recipes.head()

In [ ]:
recipes['directions']

In [ ]:
cooking_methods_glossary = [
    "bake", "steam", "fry", "grill", "roast", "boil", "saut�", "poach", "broil", "braise",
    "stew", "smoke", "microwave", "blanch", "deep-fry", "barbecue", "sear", "pressure-cook",
    "simmer", "stir-fry","baste","batter","beat","blend","carmelize","chop","cream","cube",
    "cure","dice","dissolve","drain","fold","granish","grate","grease","julienne","knead",
    "marinate","mash","mince","parboil","pare","peel","pinch","pit","plump","preheat","puree",
    "reduce","saute","scald","sear","shred","sift","skim","slice","thaw","toss","whip"
]

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
cooking_methods = []

for directions in recipes['directions']:
    if pd.isna(directions):
        cooking_methods.append(None)
    else:
        # Tokenize words
        words = word_tokenize(directions.lower())
        # Remove stopwords and non-alphabetic tokens
        filtered_words = [word for word in words if word not in stop_words and word.isalpha()]

        methods = set(filtered_words).intersection(cooking_methods_glossary)

        cooking_methods.append(list(methods))

recipes['cooking_methods'] = cooking_methods

In [ ]:
recipes['cooking_methods'].head(20)

## Clean rating dataset

In [ ]:
train_rating= pd.read_csv('core-data-train_rating.csv')
test_rating = pd.read_csv('core-data-test_rating.csv')

In [ ]:
#mapping train data with correct recipe id
unique_old_ids = sorted(train_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist() 

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
train_rating['recipe_id'] = train_rating['recipe_id'].map(mapping)

In [ ]:
#mapping test data with correct recipe id
unique_old_ids = sorted(test_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist()  

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
test_rating['recipe_id'] = test_rating['recipe_id'].map(mapping)

In [ ]:
train_rating.drop(columns=['dateLastModified'], inplace=True)
test_rating.drop(columns=['dateLastModified'], inplace=True)

In [ ]:
recipes.to_csv('cleanedrecipes.csv', index=False)

In [ ]:
train_rating.to_csv('cleanedTrainRating.csv',index=False)
test_rating.to_csv('cleanedTestRating.csv',index=False)

# Models

In [6]:
cleaned_recipes = pd.read_csv('/kaggle/input/late-plate/cleanedrecipes.csv')
cleaned_train_rating = pd.read_csv('/kaggle/input/rating/cleanedTrainRating.csv')
cleaned_test_rating = pd.read_csv('/kaggle/input/rating/cleanedTestRating.csv')

In [ ]:
# Ensure every user-recipe pair is unique
cleaned_train_rating.duplicated(subset=['user_id' , 'recipe_id']).sum()

## Collaborative Filtering

In [8]:
train_df = cleaned_train_rating.copy()
test_df = cleaned_test_rating.copy()

# Filter users with at least 5 interactions
user_counts = train_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 5].index
train_df = train_df[train_df['user_id'].isin(valid_users)]
test_df = test_df[test_df['user_id'].isin(valid_users)]

# Create binary labels based on threshold
THRESHOLD = 2  
train_df['label'] = (train_df['rating'] >= THRESHOLD).astype(int)
test_df['label'] = (test_df['rating'] >= THRESHOLD).astype(int)

# Split train into train and validation
train_data, val_data = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42,
    stratify=train_df['user_id']
)

# Combine all data for consistent indexing
combined = pd.concat([train_data, val_data, test_df])
combined['user_idx'] = combined['user_id'].astype('category').cat.codes
combined['recipe_idx'] = combined['recipe_id'].astype('category').cat.codes

# Split back to datasets
train_data = combined[combined.index.isin(train_data.index)]
val_data = combined[combined.index.isin(val_data.index)]
test_df = combined[combined.index.isin(test_df.index)]

# Get unique counts
n_users = combined['user_idx'].nunique()
n_recipes = combined['recipe_idx'].nunique()

print(f"Users: {n_users}, Recipes: {n_recipes}")
print(f"Train size: {len(train_data)}, Val size: {len(val_data)}, Test size: {len(test_df)}")


Users: 26263, Recipes: 36554
Train size: 601076, Val size: 150303, Test size: 324463


In [4]:
def build_neumf_model(n_users, n_recipes, embedding_dim=64):
    # Inputs
    user_input = Input(shape=(1,))
    recipe_input = Input(shape=(1,))
    
    # MF Path
    mf_user_embed = Embedding(n_users, embedding_dim, embeddings_regularizer=l2(0.001))(user_input)
    mf_recipe_embed = Embedding(n_recipes, embedding_dim, embeddings_regularizer=l2(0.001))(recipe_input)
    mf_user = Flatten()(mf_user_embed)
    mf_recipe = Flatten()(mf_recipe_embed)
    mf_vector = Multiply()([mf_user, mf_recipe])
    
    # MLP Path
    mlp_user_embed = Embedding(n_users, embedding_dim*2, embeddings_regularizer=l2(0.001))(user_input)
    mlp_recipe_embed = Embedding(n_recipes, embedding_dim*2, embeddings_regularizer=l2(0.001))(recipe_input)
    mlp_vector = Concatenate()([Flatten()(mlp_user_embed), Flatten()(mlp_recipe_embed)])
    mlp_vector = Dense(64, activation='relu')(mlp_vector)
    mlp_vector = BatchNormalization()(mlp_vector)
    mlp_vector = Dropout(0.3)(mlp_vector)
    
    # Combine paths
    concat = Concatenate()([mf_vector, mlp_vector])
    concat = Dense(32, activation='relu')(concat)
    
    # Output with sigmoid activation for binary classification
    output = Dense(1, activation='sigmoid')(concat)
    
    model = Model(inputs=[user_input, recipe_input], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

# Build and train model
embedding_dim = 128
model = build_neumf_model(n_users, n_recipes, embedding_dim)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 1)              │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_1             │ (None, 1)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_2 (Embedding)   │ (None, 1, 256)         │      6,723,328 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_3 (Embedding)   │ (None, 1, 256)         │      9,357,824 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_2 (Flatten)       │ (None, 256)            │              0 │ embedding_2[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_3 (Flatten)       │ (None, 256)            │              0 │ embedding_3[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 512)            │              0 │ flatten_2[0][0],       │
│                           │                        │                │ flatten_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding (Embedding)     │ (None, 1, 128)         │      3,361,664 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_1 (Embedding)   │ (None, 1, 128)         │      4,678,912 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 64)             │         32,832 │ concatenate[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten (Flatten)         │ (None, 128)            │              0 │ embedding[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_1 (Flatten)       │ (None, 128)            │              0 │ embedding_1[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 64)             │            256 │ dense[0][0]            │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multiply (Multiply)       │ (None, 128)            │              0 │ flatten[0][0],         │
│                           │                        │                │ flatten_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 64)             │              0 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_1             │ (None, 192)            │              0 │ multiply[0][0],        │
│ (Concatenate)             │                        │                │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 32)             │          6,176 │ concatenate_1[0][0]    │
├──────────────────────

 Total params: 24,161,025 (92.17 MB)

 Trainable params: 24,160,897 (92.17 MB)

 Non-trainable params: 128 (512.00 B)

In [5]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_auc', mode='max'),
    ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-5, verbose=1)
]

history = model.fit(
    [train_data['user_idx'], train_data['recipe_idx']],
    train_data['label'],
    batch_size=512,
    epochs=50,
    validation_data=([val_data['user_idx'], val_data['recipe_idx']], val_data['label']),
    callbacks=callbacks,
    verbose=1
)


Epoch 1/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.9709 - auc: 0.5603 - loss: 1.5498 - val_accuracy: 0.9855 - val_auc: 0.6511 - val_loss: 0.1567 - learning_rate: 0.0010
Epoch 2/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9861 - auc: 0.6698 - loss: 0.1511 - val_accuracy: 0.9855 - val_auc: 0.6836 - val_loss: 0.1436 - learning_rate: 0.0010
Epoch 3/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9857 - auc: 0.7626 - loss: 0.1447 - val_accuracy: 0.9855 - val_auc: 0.7190 - val_loss: 0.1213 - learning_rate: 0.0010
Epoch 4/50
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9856 - auc: 0.8369 - loss: 0.1384 - val_accuracy: 0.9855 - val_auc: 0.7193 - val_loss: 0.1259 - learning_rate: 0.0010
Epoch 5/50
1172/1174 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9859 - auc: 0.8690 - loss: 0.1296
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9859 - auc: 0.8689 

In [71]:
user_seen = combined.groupby('user_idx')['recipe_idx'].apply(set).to_dict()
all_items = np.array(combined['recipe_idx'].unique())

# Only consider positive interactions for test
test_positives = test_df[test_df['label'] == 1]
test_users = test_positives['user_idx'].unique()
def evaluate_user_level(model, k_list=[5, 10, 20]):
    
    test_user_positives = test_positives.groupby('user_idx')['recipe_idx'].apply(set).to_dict()
    combined_all = pd.concat([train_data, val_data])
    user_seen = combined_all.groupby('user_idx')['recipe_idx'].apply(set).to_dict()
    
    precision_at_k = {k: [] for k in k_list}
    recall_at_k = {k: [] for k in k_list}
    ndcg_at_k = {k: [] for k in k_list}
    
    for user in tqdm(test_users, desc="User-Level Evaluation"):
        # Get user's positive items in test set
        true_positives = test_user_positives.get(user, set())
        if not true_positives:
            continue
            
        # Get candidate items (unseen by user)
        seen = user_seen.get(user, set())
        candidate_items = list(set(all_items) - seen)
        
        negatives = list(set(all_items) - seen - true_positives)  
        if len(negatives) > 50: 
            negatives = np.random.choice(negatives, 50, replace=False)
        test_items = list(true_positives) + list(negatives)
            
        
        
        users_arr = np.full(len(test_items), user, dtype=np.int32)
        items_arr = np.array(test_items, dtype=np.int32)
        
        # Predict scores
        scores = model.predict([users_arr, items_arr], 
                              batch_size=1024, 
                              verbose=0).flatten()
        
        # Create item-score mapping
        item_scores = dict(zip(test_items, scores))
        
        # Rank all items by score (highest first)
        ranked_items = [item for item, score in 
                       sorted(item_scores.items(), key=lambda x: x[1], reverse=True)]
       
        # Calculate metrics for each K
        for k in k_list:
            
            top_k = ranked_items[:k]
            
            # Calculate true positives in top-k
            hits = len(set(top_k) & true_positives)
            precision = hits / k
            precision_at_k[k].append(precision)
            
            
            recall = hits / len(true_positives) if len(true_positives) > 0 else 0
            recall_at_k[k].append(recall)
            
            # NDCG@k
            dcg = 0
            for i, item in enumerate(top_k, 1):
                if item in true_positives:
                    dcg += 1 / np.log2(i + 1)
                    
            # Ideal DCG
            ideal_top_k = min(len(true_positives), k)
            idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal_top_k + 1))
            
            ndcg = dcg / idcg if idcg > 0 else 0
            ndcg_at_k[k].append(ndcg)
    
    
    results = {}
    for k in k_list:
        results[f'Precision@{k}'] = np.mean(precision_at_k[k])
        results[f'Recall@{k}'] = np.mean(recall_at_k[k])
        results[f'NDCG@{k}'] = np.mean(ndcg_at_k[k])   
    return results

# Run user-level evaluation
top_k = [5, 10, 20]
user_level_results = evaluate_user_level(model, k_list=top_k)

print("\n===== User-Level Evaluation Results =====")
print(f"Evaluated {len(test_users)} users")
print("-" * 40)
for k in top_k:
    print(f"** Top-{k} **")
    print(f"Precision@{k}: {user_level_results[f'Precision@{k}']:.4f}")
    print(f"Recall@{k}: {user_level_results[f'Recall@{k}']:.4f}")
    print(f"NDCG@{k}: {user_level_results[f'NDCG@{k}']:.4f}")
    print("-" * 40)


User-Level Evaluation: 100%|██████████| 26119/26119 [38:48<00:00, 11.22it/s]  



===== User-Level Evaluation Results =====
Evaluated 26119 users
----------------------------------------
** Top-5 **
Precision@5: 0.3177
Recall@5: 0.2309
NDCG@5: 0.3712
----------------------------------------
** Top-10 **
Precision@10: 0.2836
Recall@10: 0.3906
NDCG@10: 0.4006
----------------------------------------
** Top-20 **
Precision@20: 0.2383
Recall@20: 0.6104
NDCG@20: 0.4590
----------------------------------------


In [38]:
#rmse
test_pred_probs = model.predict(
    [test_df['user_idx'], test_df['recipe_idx']],
    batch_size=1024,
    verbose=1
).flatten()


binary_rmse = np.sqrt(mean_squared_error(test_df['label'], test_pred_probs))
print(f"Binary RMSE: {binary_rmse:.4f}")

317/317 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Binary RMSE: 0.1135


In [6]:
model.save("neumf_model.h5")

In [38]:
# Build mappings from entire dataset
all_users = combined['user_id'].unique()
all_recipes = combined['recipe_id'].unique()

user2idx = {user_id: idx for idx, user_id in enumerate(all_users)}
recipe2idx = {recipe_id: idx for idx, recipe_id in enumerate(all_recipes)}
idx2user = {idx: user_id for user_id, idx in user2idx.items()}
idx2recipe = {idx: recipe_id for recipe_id, idx in recipe2idx.items()}

# Create user_seen from training data
user_seen = train_data.groupby('user_idx')['recipe_idx'].apply(set).to_dict()

# Save mappings
with open('mappings.pkl', 'wb') as f:
    pickle.dump({
        'user2idx': user2idx,
        'recipe2idx': recipe2idx,
        'idx2user': idx2user,
        'idx2recipe': idx2recipe,
        'user_seen': user_seen
    }, f)

In [3]:
MODEL_PATH = "/kaggle/working/neumf_model.h5"
MAPPINGS_PATH = "/kaggle/working/mappings.pkl"

def recommend_recipes_CF(user_id, top_k=10):
    
    model = load_model(MODEL_PATH)

    
    with open(MAPPINGS_PATH, 'rb') as f:
        data = pickle.load(f)
        user2idx = data['user2idx']
        idx2recipe = data['idx2recipe']
        user_seen = data['user_seen']

    all_items = np.array(list(set().union(*user_seen.values())))  

    # Check user
    if user_id not in user2idx:
        return []  

    user_idx = user2idx[user_id]
    seen_items = user_seen.get(user_idx, set())
    candidate_items = np.setdiff1d(all_items, list(seen_items))

    if len(candidate_items) == 0:
        return []  

    # Predict
    user_input = np.full(len(candidate_items), user_idx, dtype=np.int32)
    predictions = model.predict([user_input, candidate_items], batch_size=512, verbose=0).flatten()

    # Get top-k indices and scores
    top_k_indices = np.argsort(predictions)[-top_k:][::-1]
    top_recipe_idxs = candidate_items[top_k_indices]
    top_scores = predictions[top_k_indices]
    
    
    recommendations = {
        idx2recipe[i]: float(score)  
        for i, score in zip(top_recipe_idxs, top_scores)
    }
    
    return recommendations

In [4]:
recommended = recommend_recipes_CF(user_id=3023108, top_k=20)
print("Recommended Recipes:", recommended)


Recommended Recipes: {28903: 0.9987748265266418, 4272: 0.9985865354537964, 9010: 0.9985503554344177, 27632: 0.9984862804412842, 4605: 0.998431384563446, 791: 0.9984287619590759, 9248: 0.9984036087989807, 28813: 0.998394787311554, 1250: 0.99837327003479, 4046: 0.9983643889427185, 19693: 0.998336672782898, 9013: 0.9983130693435669, 30411: 0.9982978701591492, 13180: 0.9981890320777893, 21293: 0.998187243938446, 20535: 0.9981702566146851, 22472: 0.9981689453125, 12962: 0.9981546998023987, 8082: 0.9981300234794617, 9281: 0.9981138706207275}


## Popularity Based

In [12]:
# Calculate average rating and count
popularity_avg = cleaned_train_rating.groupby('recipe_id')['rating'].agg(['mean', 'count']).reset_index()
popularity_avg.columns = ['recipe_id', 'avg_rating', 'num_ratings']

# Filter recipes with at least 3 ratings 
min_ratings = 3
popularity_avg_filtered = popularity_avg[popularity_avg['num_ratings'] >= min_ratings]

#Rank by highest average rating
top_rated = popularity_avg_filtered.sort_values(by='avg_rating', ascending=False)
print("\nHighest Average-Rated Recipes (with min 2 ratings):")
print(top_rated.head())


Highest Average-Rated Recipes (with min 2 ratings):
       recipe_id  avg_rating  num_ratings
28873      28899         5.0            4
28936      28962         5.0            3
28871      28897         5.0            3
29088      29114         5.0            4
28826      28852         5.0            3


In [13]:
#Weighted score 
min_ratings_for_weight = 1 
popularity_avg['weighted_score'] = (popularity_avg['avg_rating'] * popularity_avg['num_ratings']) / (popularity_avg['num_ratings'] + min_ratings_for_weight)

#Rank by weighted score
top_hybrid = popularity_avg.sort_values(by='weighted_score', ascending=False)
print("\nHybrid Popularity (Weighted Score):")
print(top_hybrid.head())


Hybrid Popularity (Weighted Score):
       recipe_id  avg_rating  num_ratings  weighted_score
27167      27192    4.952941           85        4.895349
19768      19790    4.906897          290        4.890034
5584        5592    4.901163          344        4.886957
19090      19111    4.904762          252        4.885375
4367        4373    4.913669          139        4.878571


In [14]:
def recommend_popular_recipes(popularity_df, n=10):
    return popularity_df['recipe_id'].tolist()[:n]

recommendations = recommend_popular_recipes(top_hybrid)
print("\nTop Recommendations:", recommendations)


Top Recommendations: [27192, 19790, 5592, 19111, 4373, 21627, 2280, 9762, 6162, 23855]


## Content Based

In [8]:
cb_recipes = cleaned_recipes.copy()  
cb_train_rating = cleaned_train_rating.copy()  

In [9]:
np.random.seed(42)  
tf.random.set_seed(42)  

physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available:", len(physical_devices))
if len(physical_devices) > 0:
    print("GPU detected:", physical_devices)
else:
    print("No GPU detected. Running on CPU.")

if tf.config.list_physical_devices('GPU'):
    set_global_policy('mixed_float16')

cb_recipes['raw_ingredients'] = cb_recipes['raw_ingredients'].fillna('')
cb_recipes['cooking_methods'] = cb_recipes['cooking_methods'].fillna('')

def parse_list(x):
    if isinstance(x, str) and x.startswith('['):
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else []
        except:
            return []
    return []

cb_recipes['raw_ingredients'] = cb_recipes['raw_ingredients'].apply(parse_list)
cb_recipes['cooking_methods'] = cb_recipes['cooking_methods'].apply(parse_list)

cb_recipes = cb_recipes[
    ~((cb_recipes['raw_ingredients'].apply(len) == 0) & (cb_recipes['cooking_methods'].apply(len) == 0)) &
    (cb_recipes['cooking_methods'].apply(len) > 0)
]
cb_recipes = cb_recipes.reset_index(drop=True)

cb_recipes['combined_text'] = cb_recipes['raw_ingredients'].apply(lambda x: ' '.join(x)) + ' ' + cb_recipes['cooking_methods'].apply(lambda x: ' '.join(x))

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
recipe_features = tfidf.fit_transform(cb_recipes['combined_text'])
recipe_ids = cb_recipes['recipe_id'].values

joblib.dump(tfidf, '/kaggle/working/tfidf_vectorizer.pkl')
joblib.dump(recipe_features, '/kaggle/working/recipe_features_sparse.pkl')


Num GPUs Available: 2
GPU detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


['/kaggle/working/recipe_features_sparse.pkl']

In [10]:
recipe_id_to_index = {rid: idx for idx, rid in enumerate(recipe_ids)}

user_profiles = {}

for user_id in cb_train_rating['user_id'].unique():
    rated = cb_train_rating[(cb_train_rating['user_id'] == user_id) & (cb_train_rating['rating'] >= 4)]
    vectors = []
    for rid in rated['recipe_id']:
        if rid in recipe_id_to_index:
            vectors.append(recipe_features[recipe_id_to_index[rid]])
    if vectors:
        avg_vector = sum(vectors) / len(vectors)
        user_profiles[user_id] = avg_vector.toarray().flatten()


In [11]:
def build_subnetwork(input_dim, num_outputs):
    return tf.keras.Sequential([
        layers.Dense(256, activation='relu', input_shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_outputs, activation='linear')
    ])


def build_dot_product_model(user_input_dim, item_input_dim, num_outputs=64):
    user_input = Input(shape=(user_input_dim,))
    item_input = Input(shape=(item_input_dim,))

    user_net = build_subnetwork(user_input_dim, num_outputs)
    item_net = build_subnetwork(item_input_dim, num_outputs)

    user_latent = user_net(user_input)
    item_latent = item_net(item_input)

    dot_output = layers.Dot(axes=1)([user_latent, item_latent])
    output = layers.Activation('sigmoid')(dot_output)

    model = Model(inputs=[user_input, item_input], outputs=output)
    return model

model2 = build_dot_product_model(user_input_dim=5000, item_input_dim=5000, num_outputs=64)
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='AUC')])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
def batch_generator(user_profiles, recipe_features, ratings_df, recipe_id_to_index, batch_size=128):
    while True:
        sampled_rows = ratings_df.sample(n=batch_size)
        user_batch = []
        recipe_batch = []
        rating_batch = []

        for _, row in sampled_rows.iterrows():
            user_id = row['user_id']
            recipe_id = row['recipe_id']
            rating = row['rating']

            if recipe_id in recipe_id_to_index and user_id in user_profiles:
                recipe_vector = recipe_features[recipe_id_to_index[recipe_id]].toarray().flatten()
                user_vector = user_profiles[user_id]

                user_batch.append(user_vector)
                recipe_batch.append(recipe_vector)
                rating_batch.append(rating)

        if user_batch:
            yield (np.array(user_batch), np.array(recipe_batch)), np.array(rating_batch)


In [13]:
def get_tf_dataset(generator_fn, user_profiles, recipe_features, ratings_df, recipe_id_to_index, batch_size):
    return tf.data.Dataset.from_generator(
        lambda: generator_fn(user_profiles, recipe_features, ratings_df, recipe_id_to_index, batch_size),
        output_signature=(
            (
                tf.TensorSpec(shape=(None, 5000), dtype=tf.float32),  # user_input
                tf.TensorSpec(shape=(None, 5000), dtype=tf.float32),  # recipe_input
            ),
            tf.TensorSpec(shape=(None,), dtype=tf.float32)  # rating
        )
    )


In [14]:
batch_size =128
train_df, val_df = train_test_split(cb_train_rating, test_size=0.2, random_state=42)
# Ratings 4 and 5 are treated as positive (1), others as negative (0)
train_df['rating'] = train_df['rating'].apply(lambda x: 1 if x >= 4 else 0)
val_df['rating'] = val_df['rating'].apply(lambda x: 1 if x >= 4 else 0)

train_dataset = get_tf_dataset(batch_generator, user_profiles, recipe_features, train_df, recipe_id_to_index, batch_size)
val_dataset = get_tf_dataset(batch_generator, user_profiles, recipe_features, val_df, recipe_id_to_index, batch_size)

steps_per_epoch = len(train_df) // batch_size
val_steps = len(val_df) // batch_size

if steps_per_epoch == 0: steps_per_epoch = 1
if val_steps == 0: val_steps = 1

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

model2.fit(
    train_dataset,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_dataset,
    validation_steps=val_steps,
    epochs=20,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 277s 64ms/step - AUC: 0.6532 - loss: 0.3433 - val_AUC: 0.7037 - val_loss: 0.3285
Epoch 2/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 190s 45ms/step - AUC: 0.7361 - loss: 0.3081 - val_AUC: 0.7152 - val_loss: 0.3208
Epoch 3/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 184s 44ms/step - AUC: 0.7706 - loss: 0.2956 - val_AUC: 0.7330 - val_loss: 0.3197
Epoch 4/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 184s 43ms/step - AUC: 0.7929 - loss: 0.2867 - val_AUC: 0.7452 - val_loss: 0.3191
Epoch 5/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 184s 43ms/step - AUC: 0.8108 - loss: 0.2753 - val_AUC: 0.7527 - val_loss: 0.3119
Epoch 6/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 182s 43ms/step - AUC: 0.8281 - loss: 0.2664 - val_AUC: 0.7561 - val_loss: 0.3121
Epoch 7/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 184s 43ms/step - AUC: 0.8390 - loss: 0.2605 - val_AUC: 0.7569 - val_loss: 0.3167
Epoch 8/20
4230/4230 ━━━━━━━━━━━━━━━━━━━━ 183s 43ms/step - AUC: 0.8503 - loss: 0.2535 - val_AUC: 0.7607 - val_loss: 0.3187
Epoch 8: early s

In [15]:
model2.save("dual_nn_model.h5")

In [16]:
recipe_id_to_index = {
    rid: idx for idx, rid in enumerate(cb_recipes['recipe_id'].values)
}
def build_user_profiles(train_df, recipe_features, recipe_id_to_index):
    user_profiles = {}
    
    for user_id in train_df['user_id'].unique():
        user_ratings = train_df[train_df['user_id'] == user_id]
        liked_items = user_ratings[user_ratings['rating'] >= 4]['recipe_id']

        vectors = []
        for rid in liked_items:
            if rid in recipe_id_to_index:
                idx = recipe_id_to_index[rid]
                vectors.append(recipe_features[idx].toarray().flatten())

        if vectors:
            user_profiles[user_id] = np.mean(vectors, axis=0)

    return user_profiles
recipe_features = joblib.load('/kaggle/working/recipe_features_sparse.pkl')
user_profiles = build_user_profiles(cb_train_rating, recipe_features, recipe_id_to_index)
joblib.dump(user_profiles, '/kaggle/working/user_profiles.pkl')

['/kaggle/working/user_profiles.pkl']

In [17]:
recipe_id_to_index = {
    rid: idx for idx, rid in enumerate(cb_recipes['recipe_id'].values)
}
def get_recommendations_dual_nn(user_id, top_k=10, batch_size=512, prefilter_top_n=2000):
    np.random.seed(42)
    random.seed(42)
    tf.random.set_seed(42)
    # Load model
    model2 = load_model("dual_nn_model.h5", compile=False)

    # Load vectorizer and features
    tfidf = joblib.load('/kaggle/working/tfidf_vectorizer.pkl')
    recipe_features = joblib.load('/kaggle/working/recipe_features_sparse.pkl')

    # Build user profiles

    user_profiles = joblib.load('/kaggle/working/user_profiles.pkl')
    recipe_ids = cb_recipes['recipe_id'].values

    if user_id not in user_profiles:
        print(f"User {user_id} not found.")
        return {}

    user_vector = user_profiles[user_id]
    rated_recipe_ids = set(cb_train_rating[cb_train_rating['user_id'] == user_id]['recipe_id'])

    # Step 1: Pre-filter with cosine similarity (sparse-safe)
    user_vector_dense = user_vector.reshape(1, -1)
    sim_scores = cosine_similarity(user_vector_dense, recipe_features).flatten()
    top_indices = np.argpartition(sim_scores, -prefilter_top_n)[-prefilter_top_n:]
    top_indices = top_indices[np.argsort(sim_scores[top_indices])[::-1]]
    top_candidates = [recipe_ids[i] for i in top_indices if recipe_ids[i] not in rated_recipe_ids and recipe_ids[i] in recipe_id_to_index]

    # Step 2: Predict scores using the neural network in batches
    scored = []

    for i in range(0, len(top_candidates), batch_size):
        batch_ids = top_candidates[i:i + batch_size]
        batch_vectors = [recipe_features[recipe_id_to_index[rid]].toarray().flatten() for rid in batch_ids]
        recipe_batch = np.array(batch_vectors)
        user_batch = np.repeat([user_vector], len(recipe_batch), axis=0)

        preds = model2.predict((user_batch, recipe_batch), verbose=0)
        scored.extend(zip(batch_ids, preds.flatten()))

    # Step 3: Sort and return top-K
    scored.sort(key=lambda x: x[1], reverse=True)
    top_recipe_ids = [rid for rid, _ in scored[:top_k]]
    top_scores = [score for _, score in scored[:top_k]]

    return dict(zip(top_recipe_ids, top_scores))

# Example usage:
recommended_recipes = get_recommendations_dual_nn(user_id=5215572, top_k=10)
print("Recommended Recipes:", recommended_recipes)


Recommended Recipes: {87568: 1.0, 575189: 1.0, 121547: 1.0, 1951776: 1.0, 624567: 1.0, 61087: 1.0, 468632: 1.0, 1444385: 1.0, 1156800: 1.0, 696818: 1.0}


In [31]:
def evaluate_user_level_dual_nn(
    model,
    train_df,
    test_df,
    user_profiles,
    recipe_features,
    recipe_ids,
    recipe_id_to_index,
    k_list=[5, 10, 20],
    negatives_per_user=50,
    sample_size=None,
    seed=42
):
    import random
    from collections import defaultdict

    np.random.seed(seed)
    random.seed(seed)
    
    precision_at_k = defaultdict(list)
    recall_at_k = defaultdict(list)
    ndcg_at_k = defaultdict(list)

    all_items_set = set(recipe_ids)
    test_df = test_df[test_df['rating'] >= 2]
    test_user_positives = test_df.groupby('user_id')['recipe_id'].apply(set).to_dict()
    user_seen = train_df.groupby('user_id')['recipe_id'].apply(set).to_dict()

    test_users = test_df['user_id'].unique()
    
    # Apply sampling if needed
    if sample_size is not None and sample_size < len(test_users):
        test_users = np.random.choice(test_users, size=sample_size, replace=False)

    for user_id in tqdm(test_users, desc="User-Level Evaluation"):
        if user_id not in user_profiles:
            continue

        true_positives = test_user_positives.get(user_id, set())
        if not true_positives:
            continue

        seen_items = user_seen.get(user_id, set())
        candidate_items = list(all_items_set - seen_items)

        negatives = list(set(candidate_items) - true_positives)
        if len(negatives) > negatives_per_user:
            negatives = np.random.choice(negatives, negatives_per_user, replace=False).tolist()

        test_items = list(true_positives) + negatives
        item_vectors = []
        item_ids = []

        for rid in test_items:
            idx = recipe_id_to_index.get(rid)
            if idx is not None and idx < recipe_features.shape[0]:
                vec = recipe_features[idx].toarray().flatten()
                item_vectors.append(vec)
                item_ids.append(rid)

        if not item_vectors:
            continue

        user_vector = user_profiles[user_id]
        user_batch = np.repeat([user_vector], len(item_vectors), axis=0)
        item_batch = np.array(item_vectors, dtype=np.float32)

        predictions = model.predict((user_batch, item_batch), verbose=0).flatten()
        scored_items = list(zip(item_ids, predictions))
        ranked_items = [item for item, _ in sorted(scored_items, key=lambda x: x[1], reverse=True)]

        for k in k_list:
            top_k = ranked_items[:k]
            hits = len(set(top_k) & true_positives)

            precision = hits / k
            recall = hits / len(true_positives)
            precision_at_k[k].append(precision)
            recall_at_k[k].append(recall)

            # NDCG
            dcg = sum([1 / np.log2(i + 2) for i, item in enumerate(top_k) if item in true_positives])
            ideal_dcg = sum([1 / np.log2(i + 2) for i in range(min(len(true_positives), k))])
            ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0
            ndcg_at_k[k].append(ndcg)

    results = {}
    for k in k_list:
        results[f'Precision@{k}'] = np.mean(precision_at_k[k])
        results[f'Recall@{k}'] = np.mean(recall_at_k[k])
        results[f'NDCG@{k}'] = np.mean(ndcg_at_k[k])
    return results


In [32]:
model2 = load_model("dual_nn_model.h5", compile=False)
tfidf = joblib.load('/kaggle/working/tfidf_vectorizer.pkl')
recipe_features = joblib.load('/kaggle/working/recipe_features_sparse.pkl')
user_profiles = joblib.load('/kaggle/working/user_profiles.pkl')
recipe_ids = cb_recipes['recipe_id'].values

top_k = [5, 10, 20]

results = evaluate_user_level_dual_nn(
    model=model2,
    train_df=cb_train_rating,
    test_df=cleaned_test_rating,
    user_profiles=user_profiles,
    recipe_features=recipe_features,
    recipe_ids=recipe_ids,
    recipe_id_to_index=recipe_id_to_index,
    k_list=top_k,
    sample_size=1000,  
    negatives_per_user=50
)

print("\n===== User-Level Evaluation Results =====")
for k in top_k:
    print(f"Top-{k} → Precision: {results[f'Precision@{k}']:.4f}, Recall: {results[f'Recall@{k}']:.4f}, NDCG: {results[f'NDCG@{k}']:.4f}")



User-Level Evaluation: 100%|██████████| 1000/1000 [08:07<00:00,  2.05it/s]


===== User-Level Evaluation Results =====
Top-5 → Precision: 0.0581, Recall: 0.0948, NDCG: 0.0917
Top-10 → Precision: 0.0531, Recall: 0.1651, NDCG: 0.1160
Top-20 → Precision: 0.0503, Recall: 0.3093, NDCG: 0.1599


## Hybrid Filtering

In [30]:
model2 = load_model("/kaggle/working/dual_nn_model.h5", compile=False)
tfidf = joblib.load('/kaggle/working/tfidf_vectorizer.pkl')
recipe_features = joblib.load('/kaggle/working/recipe_features_sparse.pkl')
user_profiles = joblib.load('/kaggle/working/user_profiles.pkl')
MODEL_PATH = "/kaggle/working/neumf_model.h5"
MAPPINGS_PATH = "/kaggle/working/mappings.pkl"
model = load_model(MODEL_PATH)

    
with open('mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
    user2idx = mappings['user2idx']
    recipe2idx =mappings['recipe2idx']
    idx2user = mappings['idx2user']
    idx2recipe = mappings['idx2recipe']
    user_seen = mappings['user_seen']
# Create inverse mappings
idx2user = {idx: user_id for user_id, idx in user2idx.items()}  # CF user index -> Original user ID
all_items = np.array(list(idx2recipe.keys()))  # All CF recipe indices

In [31]:
test_positives = test_df[test_df['label'] == 1]
test_users = test_positives['user_idx'].unique()
test_user_positives = test_positives.groupby('user_idx')['recipe_idx'].apply(set).to_dict()


In [36]:
def hybrid_evaluate_user_level(cf_model, cb_model, alpha,
                              k_list=[5, 10, 20]):
   with open('mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
    user2idx = mappings['user2idx']
    recipe2idx =mappings['recipe2idx']
    idx2user = mappings['idx2user']
    idx2recipe = mappings['idx2recipe']
    user_seen = mappings['user_seen']
    
    # Create inverse user mapping
    idx2user = {idx: user_id for user_id, idx in user2idx.items()}
    
    # Get all known recipe indices
    known_recipes = set(idx2recipe.keys())
    
    # Prepare test data
    test_positives = test_df[test_df['label'] == 1]
    test_users = test_positives['user_idx'].unique()
    test_user_positives = test_positives.groupby('user_idx')['recipe_idx'].apply(set).to_dict()
    
    precision_at_k = {k: [] for k in k_list}
    recall_at_k = {k: [] for k in k_list}
    ndcg_at_k = {k: [] for k in k_list}
    
    for user in tqdm(test_users, desc="Hybrid Evaluation"):
        # Get true positives that exist in our mapping
        true_positives = test_user_positives.get(user, set())
        valid_true_positives = true_positives & known_recipes
        if not valid_true_positives:
            continue
            
        # Get original user ID for CB
        original_user_id = idx2user.get(user)
        if not original_user_id or original_user_id not in user_profiles:
            continue
            
        # Generate candidate items (only known recipes)
        seen = user_seen.get(user, set())
        candidate_items = list(known_recipes - seen)
        
        # Create negatives from known recipes
        negatives = list(known_recipes - seen - valid_true_positives)
        if len(negatives) > 50:
            negatives = np.random.choice(negatives, 50, replace=False)
        test_items = list(valid_true_positives) + list(negatives)
        
        # Get CF predictions
        user_arr = np.full(len(test_items), user)
        item_arr = np.array(test_items)
        cf_scores = cf_model.predict([user_arr, item_arr], verbose=0).flatten()
        
        # Get CB predictions with safe recipe ID conversion
        candidate_recipe_ids = []
        for idx in test_items:
            try:
                candidate_recipe_ids.append(idx2recipe[idx])
            except KeyError:
                # Shouldn't happen since we filtered, but just in case
                candidate_recipe_ids.append(None)
        
        recipe_vecs = []
        for rid in candidate_recipe_ids:
            if rid is not None and rid in recipe2idx:
                vec = recipe_features[recipe2idx[rid]].toarray().flatten()
            else:
                # Use zero vector for missing recipes
                vec = np.zeros(recipe_features.shape[1])
            recipe_vecs.append(vec)
        
        user_vec = np.tile(user_profiles[original_user_id], (len(recipe_vecs), 1))
        cb_scores = cb_model.predict([user_vec, np.array(recipe_vecs)], verbose=0).flatten()
        
        # Normalize CB scores to [0,1]
        cb_min, cb_max = np.min(cb_scores), np.max(cb_scores)
        if cb_max > cb_min:  # Avoid division by zero
            cb_scores = (cb_scores - cb_min) / (cb_max - cb_min)
        else:
            cb_scores = np.zeros_like(cb_scores)
        
        # Combine scores
        hybrid_scores = alpha * cf_scores + (1 - alpha) * cb_scores
        item_scores = dict(zip(test_items, hybrid_scores))
        
        # Rank items
        ranked_items = [item for item, score in 
                      sorted(item_scores.items(), key=lambda x: x[1], reverse=True)]
        
        # Calculate metrics using valid true positives
        for k in k_list:
            top_k = ranked_items[:k]
            hits = len(set(top_k) & valid_true_positives)
            
            # Precision
            precision_at_k[k].append(hits / k)
            
            # Recall
            recall = hits / len(valid_true_positives) if valid_true_positives else 0
            recall_at_k[k].append(recall)
            
            # NDCG
            dcg = 0
            for i, item in enumerate(top_k, 1):
                if item in valid_true_positives:
                    dcg += 1 / np.log2(i + 1)
            
            ideal_top_k = min(len(valid_true_positives), k)
            idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal_top_k + 1))
            
            ndcg = dcg / idcg if idcg > 0 else 0
            ndcg_at_k[k].append(ndcg)
    
    # Compile results
    results = {}
    for k in k_list:
        results[f'Precision@{k}'] = np.mean(precision_at_k[k]) if precision_at_k[k] else 0
        results[f'Recall@{k}'] = np.mean(recall_at_k[k]) if recall_at_k[k] else 0
        results[f'NDCG@{k}'] = np.mean(ndcg_at_k[k]) if ndcg_at_k[k] else 0
        
    return results

In [ ]:
results = hybrid_evaluate_user_level(
        cf_model=model,
        cb_model=model2,
        alpha=0.5,
        k_list=[5,10,20]
    )
    

In [34]:
print(f"Evaluated {len(test_users)} users")
for k in [5,10,20]:
    print(f"** Top-{k} **")
    print(f"Precision@{k}: {results[f'Precision@{k}']:.4f}")
    print(f"Recall@{k}:    {results[f'Recall@{k}']:.4f}")
    print(f"NDCG@{k}:      {results[f'NDCG@{k}']:.4f}")
    print("-" * 40)



Evaluated 26119 users
** Top-5 **
Precision@5: 0.3262
Recall@5:    0.2386
NDCG@5:      0.3813
----------------------------------------
** Top-10 **
Precision@10: 0.2850
Recall@10:    0.3883
NDCG@10:      0.4048
----------------------------------------
** Top-20 **
Precision@20: 0.2360
Recall@20:    0.5989
NDCG@20:      0.4585
----------------------------------------


In [22]:
def hybrid_recommend(original_user_id, cf_model, cb_model, 
                    user2idx, recipe2idx, recipe_id_to_index,
                    user_profiles, recipe_features, top_hybrid, alpha, k):
    """
    Get top-k hybrid recommendations for a user, with fallback to popular recipes.
    
    Args:
        original_user_id: Original user ID from dataset
        k: Number of recommendations to return
        alpha: Weight for CF model (0-1), CB weight = 1 - alpha
    Returns:
        List of recommended recipe IDs
    """
    # Get all recipe IDs in the system
    all_recipe_ids = list(recipe2idx.keys())
    
    # New user fallback - return top k popular recipes
    if original_user_id not in user2idx or original_user_id not in user_profiles:
        return top_hybrid['recipe_id'].tolist()[:k]
    
    # Existing user - proceed with hybrid recommendation
    user_idx = user2idx[original_user_id]
    
    # Create candidate items (all recipes not seen by user)
    seen_indices = user_seen.get(user_idx, set())
    candidate_indices = list(set(recipe2idx.values()) - seen_indices)
    
    # Get CF predictions
    user_arr = np.full(len(candidate_indices), user_idx)
    item_arr = np.array(candidate_indices)
    cf_scores = cf_model.predict([user_arr, item_arr], verbose=0).flatten()
    
    # Get CB predictions
    candidate_recipe_ids = [idx2recipe[idx] for idx in candidate_indices]
    recipe_vecs = []
    for rid in candidate_recipe_ids:
        if rid in recipe_id_to_index:
            vec = recipe_features[recipe_id_to_index[rid]].toarray().flatten()
        else:
            vec = np.zeros(recipe_features.shape[1])
        recipe_vecs.append(vec)
    
    user_vec = np.tile(user_profiles[original_user_id], (len(recipe_vecs), 1))
    cb_scores = cb_model.predict([user_vec, np.array(recipe_vecs)], verbose=0).flatten()
    
    # Min-max normalize CB scores to [0,1]
    cb_min, cb_max = cb_scores.min(), cb_scores.max()
    if cb_max > cb_min:  # Avoid division by zero
        cb_scores = (cb_scores - cb_min) / (cb_max - cb_min)
    else:
        cb_scores = np.zeros_like(cb_scores)
    
    # Combine scores
    hybrid_scores = alpha * cf_scores + (1 - alpha) * cb_scores
    
    # Get top-k recommendations
    sorted_indices = np.argsort(hybrid_scores)[::-1][:k]  # Descending order
    top_recipe_ids = [candidate_recipe_ids[i] for i in sorted_indices]
    
    return top_recipe_ids

In [29]:
# Get all unique recipe IDs in the system
all_recipe_ids = combined['recipe_id'].unique().tolist()
model2 = load_model("/kaggle/working/dual_nn_model.h5", compile=False)
model = load_model("/kaggle/working/neumf_model.h5")
tfidf = joblib.load('/kaggle/working/tfidf_vectorizer.pkl')
recipe_features = joblib.load('/kaggle/working/recipe_features_sparse.pkl')
user_profiles = joblib.load('/kaggle/working/user_profiles.pkl')
with open('mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
    user2idx = mappings['user2idx']
    recipe2idx =mappings['recipe2idx']
    idx2user = mappings['idx2user']
    idx2recipe = mappings['idx2recipe']
    user_seen = mappings['user_seen']
# For a known user
recommendations = hybrid_recommend(
    original_user_id=3023108,
    cf_model=model,
    cb_model=model2,
    user2idx=user2idx,
    recipe2idx=recipe2idx,
    recipe_id_to_index=recipe2idx,
    user_profiles=user_profiles,
    recipe_features=recipe_features,
    top_hybrid=top_hybrid,
    alpha=0.7,
    k=10
)
print("Top Recommendations:", recommendations)

Top Recommendations: [15262, 23038, 1078, 3813, 21176, 17191, 10639, 7976, 5150, 12843]
